Exploratory NLP Analysis

In [ ]:
# =============================================
# STEP 0: Install Required Libraries
# =============================================
!pip install wordcloud
!pip install nltk
!pip install textblob
!pip install gensim
!pip install pyLDAvis
!pip install matplotlib
!pip install seaborn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 26.3 MB/s eta 0:00:00


In [ ]:
# =============================================
# STEP 1: Import Libraries
# =============================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords
from textblob import TextBlob
from gensim import corpora, models
import pyLDAvis.gensim_models
import re
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Download NLTK stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))


In [ ]:
# =============================================
# STEP 2: Load Cleaned Dataset
# =============================================
df = pd.read_csv('cleaned_project_dataset.csv')


In [ ]:
# =============================================
# STEP 3: Preprocess Text
# =============================================
def preprocess_text(text):
    """Clean and tokenize text for NLP"""
    text = str(text).lower()                     # Lowercase
    text = re.sub(r'[^a-z0-9\s]', '', text)     # Remove punctuation
    tokens = text.split()                        # Tokenize by space
    tokens = [word for word in tokens if word not in stop_words]  # Remove stopwords
    return tokens

df['tokens'] = df['Project Description_Clean'].apply(preprocess_text)


In [ ]:
# =============================================
# STEP 4: Word Cloud of Most Frequent Words
# =============================================
all_words = [word for tokens in df['tokens'] for word in tokens]
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(' '.join(all_words))

plt.figure(figsize=(15,7))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("Word Cloud of Project Descriptions", fontsize=18)
plt.show()


In [ ]:
# =============================================
# STEP 5: Topic Modeling using LDA
# =============================================
# Create dictionary and corpus for LDA
dictionary = corpora.Dictionary(df['tokens'])
corpus = [dictionary.doc2bow(text) for text in df['tokens']]

# Train LDA model
lda_model = models.LdaModel(corpus=corpus, id2word=dictionary, num_topics=5, passes=15, random_state=42)

# Print topics
for idx, topic in lda_model.print_topics(-1):
    print(f"Topic {idx+1}: {topic}\n")

# Optional: Interactive visualization
pyLDAvis.enable_notebook()
lda_vis = pyLDAvis.gensim_models.prepare(lda_model, corpus, dictionary)
lda_vis


In [ ]:
# =============================================
# STEP 6: Sentiment Analysis
# =============================================
def get_sentiment(text):
    """Return polarity (-1 to 1) and classify sentiment"""
    blob = TextBlob(str(text))
    polarity = blob.sentiment.polarity
    if polarity > 0.05:
        sentiment = 'Positive'
    elif polarity < -0.05:
        sentiment = 'Negative'
    else:
        sentiment = 'Neutral'
    return pd.Series([polarity, sentiment])

df[['Polarity', 'Sentiment']] = df['Project Description_Clean'].apply(get_sentiment)

# Plot sentiment distribution
plt.figure(figsize=(8,5))
sns.countplot(data=df, x='Sentiment', order=['Positive','Neutral','Negative'])
plt.title("Sentiment Distribution of Project Descriptions", fontsize=16)
plt.show()

In [ ]:
# =============================================
# STEP 7: Correlation of Text Sentiment with Net_Profit and Status
# =============================================
# Average ROI per sentiment
roi_sentiment = df.groupby('Sentiment')['Net_Profit'].mean().reset_index()
plt.figure(figsize=(8,5))
sns.barplot(data=roi_sentiment, x='Sentiment', y='Net_Profit', palette='viridis')
plt.title("Average Net_Profit by Sentiment", fontsize=16)
plt.show()

# Check common words for Completed vs Cancelled projects
completed_text = df[df['Status'] == 'Completed']['Project Description_Clean'].str.cat(sep=' ')
cancelled_text = df[df['Status'] == 'Cancelled']['Project Description_Clean'].str.cat(sep=' ')

# Generate word clouds for Completed vs Cancelled
fig, ax = plt.subplots(1,2, figsize=(20,8))
wc_completed = WordCloud(width=800, height=400, background_color='white').generate(completed_text)
wc_cancelled = WordCloud(width=800, height=400, background_color='white').generate(cancelled_text)

ax[0].imshow(wc_completed, interpolation='bilinear')
ax[0].axis('off')
ax[0].set_title('Completed Projects', fontsize=16)

ax[1].imshow(wc_cancelled, interpolation='bilinear')
ax[1].axis('off')
ax[1].set_title('Cancelled Projects', fontsize=16)
plt.show()

# Optional: Identify top words per status
from collections import Counter

def top_words(text, n=20):
    tokens = preprocess_text(text)
    return Counter(tokens).most_common(n)

print("Top words in Completed Projects:", top_words(completed_text))
print("Top words in Cancelled Projects:", top_words(cancelled_text))
